# Supplementary Figure 1

Selected database sizes in the context of the full sequence reservoirs. Each blue point is the
number of FASTA entries in one acquisition-specific database (726 acquisitions in four cohorts);
bars are cohort medians, orange diamonds are the full cohort reservoirs, and the dashed line is the
20,431-entry human reference proteome. The axis is logarithmic.

Run from the repository root: `python paper.py build SFig01`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

## Panel a

The figure has one unlettered panel, called `a` here. It shows FASTA entries per acquisition-specific
database, per cohort, against the full reservoir sizes.

Claims: "All 726 acquisition-specific databases contained fewer entries than the 20,431-entry human
reference proteome" and "full study reservoirs containing 233.5 million sequences for healthy stool,
272.9 million for PREDICT, 264.9 million for CapScan and 375.3 million for CAMPI (Supplementary Fig. 1)"
(v28d line 55 and 57; same wording in v30).

n = 726 acquisitions (15 CAMPI, 58 healthy stool, 306 PREDICT, 347 CapScan); one reservoir per cohort.
Test: none (descriptive). Point jitter is for display only and uses seed `panels.JITTER_SEED`.

Two steps. `panel_a` is the original producer (matplotlib defaults); its output is the v28f figure.
`panel_a_v30` redraws the same values in the house style (Söhne, ink, Okabe-Ito colours by meaning,
drawn at the placed size of 180 x 105.2 mm); its output is the v30 figure. `values_unchanged_v30`
proves that every plotted value, limit and scale of the two is identical.


In [ ]:
from IPython.display import Image, display
SIZES = REPO / "source_data" / "Fig1" / "acquisition_sizes_726.tsv"   # shared with Figure 1 (lives under Fig1)
panels.check_sha256(SIZES, panels.SIZES_SHA256)
t = panels.load_sizes(SIZES)
summary = panels.panel_a(t, OUT / "panel_a.pdf")
print("per cohort (n, median, min, max):", summary)
raster = panels.render_composite(OUT / "panel_a.pdf", OUT / "SFig01_v28f.png")
print("composite raster:", raster)
import fitz
fitz.open(OUT / "panel_a.pdf")[0].get_pixmap(dpi=300, alpha=False).save(OUT / "panel_a.png")
display(Image(filename=str(OUT / "panel_a.png"), width=720))
v30 = panels.values_unchanged_v30(t, OUT / "SFig01_v30")
assert v30["values_identical"] and not v30["label_strings_lost"] and not v30["label_strings_added"]
print("v30 values identical to producer:", v30["values_identical"], "| png px:", v30["png_px"])
display(Image(filename=str(OUT / "SFig01_v30.png"), width=720))


## Checks and numbers

Pixel checks against the original producer's outputs: the panel PDF at 300 dpi, and the 400 dpi
v28f composite. The producer drew all text in matplotlib's default typeface (DejaVu Sans), which
ships with matplotlib, so these checks always run. The v30 house-style figure is compared with the
v30 PNG (`expected/SFig01_v30.png`, 300 dpi); its text is Söhne, so that check runs only when
`FASTALAKE_FONT_DIR` is set. Then every recomputed number, in `NUMBERS.tsv` order.


In [ ]:
from tools.check_pixels import compare
for new, exp, stem in [(OUT / "panel_a.pdf", UNIT / "expected" / "panel_a.pdf", "panel_a"),
                       (OUT / "SFig01_v28f.png", UNIT / "expected" / "SFig01_v28f.png", "SFig01_v28f")]:
    r = compare(new, exp, UNIT / "checks" / stem, dpi=300)
    print(stem, r["verdict"], "differing pixels:", r.get("differing_pixels"))
import os
if os.environ.get("FASTALAKE_FONT_DIR") or os.environ.get("FL_STYLE_DIR"):
    r = compare(OUT / "SFig01_v30.png", UNIT / "expected" / "SFig01_v30.png", UNIT / "checks" / "SFig01_v30", dpi=300)
    print("SFig01_v30", r["verdict"], "differing pixels:", r.get("differing_pixels"))
else:
    print("pixel checks SKIPPED: fonts absent (SFig01_v30)")

n = panels.stat_counts(t)
e = panels.stat_entries(t)
res = panels.stat_reservoirs(t)
h = panels.stat_human_reference(SD / "human_swissprot_canonical_2026_03.fasta.gz")
assert h["sha256_matches_upstream"] and h["entries"] == panels.HUMAN_REFERENCE_ENTRIES
print("human reference sha256 matches upstream:", h["sha256_matches_upstream"])
print("databases below human reference:", e["n_below_human"], "of", n["total"], "; largest:", e["max_all"])
numbers = [
    ("55", "n_below_human", e["n_below_human"]),
    ("55", "human_reference_entries", h["entries"]),
    ("55", "median_CAMPI", e["median"]["CAMPI"]),
    ("55", "median_HSTOOL", e["median"]["HSTOOL"]),
    ("57", "reservoir_HSTOOL_millions", res["HSTOOL"] / 1e6),
    ("57", "reservoir_PREDICT_millions", res["PREDICT"] / 1e6),
    ("57", "reservoir_CapScan_millions", res["CapScan"] / 1e6),
    ("57", "reservoir_CAMPI_millions", res["CAMPI"] / 1e6),
    ("57", "public_reservoir_millions (= HSTOOL)", res["HSTOOL"] / 1e6),
    ("405", "n_total", n["total"]),
    ("405", "human_reference_entries", h["entries"]),
    ("405", "public_reservoir_entries (= HSTOOL)", res["HSTOOL"]),
    ("405", "n_CAMPI", n["CAMPI"]),
    ("405", "n_HSTOOL", n["HSTOOL"]),
    ("405", "n_PREDICT", n["PREDICT"]),
    ("405", "n_CapScan", n["CapScan"]),
    ("673", "n_total", n["total"]),
    ("673", "human_reference_entries", h["entries"]),
    ("675", "n_total", n["total"]),
    ("675", "public_reservoir_entries (= HSTOOL)", res["HSTOOL"]),
]
for line, key, val in numbers:
    print(f"{line}\t{key}\t{val!r}")